# Where do the errors come from?

Two suspects:

1. **the reader** (pdfplumber) — did it damage the document?
2. **the models** — did they label it wrong?

The same three-step check is run on two documents — and they reach **opposite
verdicts**.


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json
import pandas as pd
from IPython.display import display

from dmpbridge.core import paths as P
from dmpbridge.evaluation.evaluate import (
    containment, extract_gold, resolve_old_gt_path, tokenize,
)

EXTRACTOR = 'pdfplumber'
MODELS = ['llama3.1:8b', 'gemma4:e4b', 'llama3.3:70b']
pd.set_option('display.max_colwidth', 80)


def load(n):
    """Annotation, reader blocks, and each block's true label for sample n.

    A block's true label is the label of the annotation item its text belongs
    to. A block that does not fit any single item gets None — that only
    happens when one block spans two items at once.
    """
    gold = extract_gold(resolve_old_gt_path(n))
    blocks = json.loads((P.EXTRACTED_DIR / EXTRACTOR / f'sample{n}.json')
                        .read_text(encoding='utf-8'))
    truth = []
    for b in blocks:
        bt = tokenize(b['text'])
        best, lab = 0.0, None
        for gt, gl in gold:
            c = containment(bt, tokenize(gt))
            if c > best:
                best, lab = c, gl
        truth.append(lab if best >= 0.75 else None)
    return gold, blocks, truth


def load_labels(model, n):
    """The same blocks after one model labeled them."""
    return json.loads(P.labeled_path(P.make_tag(model, EXTRACTOR), n)
                      .read_text(encoding='utf-8'))


## Sample 1


### Step 1 — Did the reader damage sample 1?

Every block the reader produced is checked against the annotation: does its
text belong to **one** real item?


In [2]:
gold, blocks, truth = load(1)

lost = [i for i, t in enumerate(truth) if t is None]
print(f'the annotation has {len(gold)} items')
print(f'the reader produced {len(blocks)} blocks')
print(f'blocks whose text does not fit any single item: {len(lost)}')
for i in lost:
    print(f'   block {i:>2}: {blocks[i]["text"][:66]!r}')


the annotation has 28 items
the reader produced 78 blocks
blocks whose text does not fit any single item: 0


### Step 2 — How well did each model label the same blocks?


In [3]:
labeled = {m: load_labels(m, 1) for m in MODELS}
known = sum(1 for t in truth if t)
acc = pd.DataFrame([
    {'model': m,
     'correct': sum(1 for t, b in zip(truth, labeled[m]) if t and b.get('label') == t),
     'wrong': sum(1 for t, b in zip(truth, labeled[m]) if t and b.get('label') != t),
     'accuracy': sum(1 for t, b in zip(truth, labeled[m]) if t and b.get('label') == t) / known}
    for m in MODELS]).set_index('model')
display(acc.style.background_gradient(cmap='Greens', subset=['accuracy'])
        .format({'accuracy': '{:.0%}'}))


,correct,wrong,accuracy
model,,,
llama3.1:8b,67,11,86%
gemma4:e4b,78,0,100%
llama3.3:70b,77,1,99%


### Step 3 — The wrong blocks, one by one


In [4]:
for m in MODELS:
    bad = [{'true label': t, 'model said': b.get('label'), 'text': b['text']}
           for t, b in zip(truth, labeled[m]) if t and b.get('label') != t]
    print(f'{m}: {len(bad)} wrong')
    if bad:
        display(pd.DataFrame(bad))
    print()


llama3.1:8b: 11 wrong


,true label,model said,text
0,question.text,section.title,"B. Scientific data that will be preserved and shared, and the rationale for ..."
1,answer.text,section.description,about other studies.
2,question.text,section.title,"C. Metadata, other relevant data, and associated documentation:"
3,answer.text,question.text,The following data will be created as a result of this project:
4,question.text,section.title,A. Repository where scientific data and metadata will be archived:
5,question.text,section.title,B. How scientific data will be findable and identifiable:
6,question.text,section.title,C. When and how long the scientific data will be made available:
7,question.text,section.title,"A. Factors affecting subsequent access, distribution, or reuse of scientific..."
8,answer.text,question.text,about using their data:
9,question.text,section.title,B. Whether access to scientific data will be controlled:



gemma4:e4b: 0 wrong

llama3.3:70b: 1 wrong


,true label,model said,text
0,answer.text,section.description,The following data will be created as a result of this project:


### Conclusion for sample 1

**The errors come from the model, not from pdfplumber.**

gemma4 received exactly the same blocks and labeled every one correctly — so the
input was good enough to score 100%, and every error here belongs to the model.

llama3.1's mistakes are mostly one repeated error: lettered lines like
`B. Scientific data that will be preserved…` are questions, but they *look* like
headings, and it calls them headings.


## Sample 6 — the opposite case

Same check, different document. This one's section headings are **underlined**
instead of bold.


### Step 1 — Did the reader damage sample 6?

Every block the reader produced is checked against the annotation: does its
text belong to **one** real item?


In [5]:
gold, blocks, truth = load(6)

lost = [i for i, t in enumerate(truth) if t is None]
print(f'the annotation has {len(gold)} items')
print(f'the reader produced {len(blocks)} blocks')
print(f'blocks whose text does not fit any single item: {len(lost)}')
for i in lost:
    print(f'   block {i:>2}: {blocks[i]["text"][:66]!r}')


the annotation has 11 items
the reader produced 24 blocks
blocks whose text does not fit any single item: 3
   block  6: '2. Data and metadata standards. The PI’s research group will adopt'
   block 13: '4. Policies and provisions for re-use, re-distribution. Simulation'
   block 18: '5. Plans for archiving and preservation of access. Local data will'


### Step 2 — How well did each model label the same blocks?


In [6]:
labeled = {m: load_labels(m, 6) for m in MODELS}
known = sum(1 for t in truth if t)
acc = pd.DataFrame([
    {'model': m,
     'correct': sum(1 for t, b in zip(truth, labeled[m]) if t and b.get('label') == t),
     'wrong': sum(1 for t, b in zip(truth, labeled[m]) if t and b.get('label') != t),
     'accuracy': sum(1 for t, b in zip(truth, labeled[m]) if t and b.get('label') == t) / known}
    for m in MODELS]).set_index('model')
display(acc.style.background_gradient(cmap='Greens', subset=['accuracy'])
        .format({'accuracy': '{:.0%}'}))


,correct,wrong,accuracy
model,,,
llama3.1:8b,9,12,43%
gemma4:e4b,16,5,76%
llama3.3:70b,18,3,86%


### Step 3 — The wrong blocks, one by one


In [7]:
for m in MODELS:
    bad = [{'true label': t, 'model said': b.get('label'), 'text': b['text']}
           for t, b in zip(truth, labeled[m]) if t and b.get('label') != t]
    print(f'{m}: {len(bad)} wrong')
    if bad:
        display(pd.DataFrame(bad))
    print()


llama3.1:8b: 12 wrong


,true label,model said,text
0,answer.text,section.title,"1. Types of data. The bulk of the data generated in this project will be 1, ..."
1,answer.text,question.text,dimensional arrays of numbers and bytes representing fluid state variables g...
2,answer.text,question.text,"data using both commercial packages, such as Matlab and free visualization p..."
3,answer.text,section.description,practice that all generated data should be straightforwardly reproducible. T...
4,answer.text,question.text,stored within the same directory on a mass storage system along with the dat...
5,answer.text,section.title,3. Policies for access and sharing. Interested parties will be able to reque...
6,answer.text,question.text,the end of the project. Such parties will be responsible for their own versi...
7,answer.text,question.text,several terrabytes with an associated cost of storage. Data will be made pub...
8,answer.text,question.text,intellectual rights to the code will be retained by the CU group. Disseminat...
9,answer.text,question.text,"mass storage system. As mentioned above, all data generated is archived with..."



gemma4:e4b: 5 wrong


,true label,model said,text
0,answer.text,section.title,"1. Types of data. The bulk of the data generated in this project will be 1, ..."
1,answer.text,section.title,3. Policies for access and sharing. Interested parties will be able to reque...
2,answer.text,question.text,the end of the project. Such parties will be responsible for their own versi...
3,answer.text,question.text,several terrabytes with an associated cost of storage. Data will be made pub...
4,answer.text,question.text,"mass storage system. As mentioned above, all data generated is archived with..."



llama3.3:70b: 3 wrong


,true label,model said,text
0,answer.text,section.title,"1. Types of data. The bulk of the data generated in this project will be 1, ..."
1,answer.text,section.title,3. Policies for access and sharing. Interested parties will be able to reque...
2,answer.text,section.title,1


### Conclusion for sample 6

**Here the reader is at fault — and no model can fix it.**

This document's headings are underlined, and an underline is a drawn line the reader
cannot see. So each heading arrives **glued to its answer in one block** — two items,
one block, one label. Whichever label the model picks, the other item is lost.

That is why *every* model struggles here, including the one that was perfect on
sample 1. When all models fail on the same blocks, the problem sits before the models.

### The overall lesson

| document | who is at fault | how you can tell |
|---|---|---|
| sample 1 | the model | one model scored 100% from the same input |
| sample 6 | the reader | every model fails on the same fused blocks |
